# RT-BENE: eyes open or closed (Kaggle)

**What this notebook does:** It lets you inspect RT-BENE data and, if you choose, train an eye-state model from VS Code using a Kaggle GPU. This covers **one part** of drowsiness detection. It does not train the distraction model.

**What is already running:** A separate [saved Kaggle training run](https://www.kaggle.com/code/mfahadwaqar2002/dlcv-driver-drowsiness-detection) trains the full model. You do **not** need to run Step 4 here while that run is running.

**Where files go:** The data and code go to Kaggle's temporary storage and disappear when the session ends. Interactive model results go to `/kaggle/working/artifacts`; download them before the session ends. The separate saved Kaggle run has its own versioned outputs.

Run Steps 1–3 in order. Step 4 is optional. Step 5 shows results if you ran Step 4.


## Step 1 — Check the connection

In VS Code, select the **Kaggle Jupyter Server** kernel. This cell confirms that the code is running on Kaggle with a GPU.


In [ ]:
from pathlib import Path
import subprocess
import sys
import torch

assert Path('/kaggle/working').is_dir(), 'Connect VS Code to the Kaggle Jupyter Server first.'
assert torch.cuda.is_available(), 'Enable a GPU in Kaggle Settings > Accelerator, then reconnect.'
print('Connected to Kaggle')
print('GPU:', torch.cuda.get_device_name(0))


## Step 2 — Load this project's code

The notebook fetches the exact GitHub commit used by the current saved run. This keeps experiments reproducible. Re-run this cell after changing the commit below.


In [ ]:
SOURCE_COMMIT = 'bc0bb0e6b7687c63db7fff6aac9d11044f451928'
repo = Path('/kaggle/temp/driver-monitor-code')
if not (repo / '.git').is_dir():
    subprocess.run(
        ['git', 'clone', 'https://github.com/mfahadwaqar/DLCV-Driver-Distraction-Detection.git', str(repo)],
        check=True,
    )
subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', SOURCE_COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == SOURCE_COMMIT
(repo / 'SOURCE_COMMIT').write_text(SOURCE_COMMIT + '\n')
print('Loaded project code:', SOURCE_COMMIT[:7])


## Step 3 — Get RT-BENE and check the labels

This downloads the official RT-BENE files **to Kaggle**, verifies their checksums, and prepares left-eye images. The first run can take several minutes. If this session already has a completed manifest, the cell reuses it. The images never need to be downloaded to your laptop.

Labels: **0 = open eyes**, **1 = closed/blinking eyes**. The classes are imbalanced, so the training code uses weights calculated from the training split only.


In [ ]:
import csv
from collections import Counter

data = Path('/kaggle/temp/rt-bene')
manifest = data / 'prepared' / 'manifest.csv'
if not manifest.is_file():
    print('Downloading and verifying RT-BENE on Kaggle...')
    subprocess.run(
        [sys.executable, '-m', 'src.data.fetch_rt_bene', '--output', str(data / 'source')],
        cwd=repo, check=True,
    )
    print('Preparing labeled eye images...')
    subprocess.run(
        [sys.executable, '-m', 'src.data.rt_bene', '--source', str(data / 'source'),
         '--output', str(data / 'prepared')],
        cwd=repo, check=True,
    )
else:
    print('Using data already prepared in this Kaggle session.')

with manifest.open(newline='', encoding='utf-8') as stream:
    rows = list(csv.DictReader(stream))
print('Labeled images:', len(rows))
print('Subjects:', len({row['subject_id'] for row in rows}))
print('Labels (0=open, 1=closed):', dict(Counter(row['label_drowsiness'] for row in rows)))


## Step 4 — Train here (optional)

**Skip this step if you only want to view data.** The separate saved Kaggle run is already doing full training. To run an additional experiment here, set `RUN_TRAINING = True` and choose a **new** `RUN_NAME`. Training uses subject-separated train, validation, and test splits. It saves a checkpoint and metrics under `/kaggle/working/artifacts`.

Interactive results can disappear when this session stops. For a durable full run, use the saved-version workflow in [the project guide](../docs/KAGGLE.md).


In [ ]:
RUN_TRAINING = False
RUN_NAME = 'rt-bene-interactive-01'

output = Path('/kaggle/working/artifacts') / RUN_NAME
splits = Path('/kaggle/working/artifacts/rt-bene-interactive-splits.json')
if RUN_TRAINING:
    subprocess.run(
        [sys.executable, '-m', 'src.train.phase1',
         '--data-root', str(data / 'prepared'),
         '--manifest', str(manifest),
         '--output', str(output),
         '--splits', str(splits),
         '--device', 'cuda'],
        cwd=repo, check=True,
    )
    print('Training finished. Results:', output)
else:
    print('Training skipped. The separate saved Kaggle run is independent of this notebook.')


## Step 5 — View this notebook's results

This checks **only** the optional interactive run above. For the separate saved run, open its Kaggle version and inspect **Output** after it finishes.


In [ ]:
import json

summary_file = output / 'summary.json'
if not summary_file.is_file():
    print('No interactive training results yet. This is expected if Step 4 was skipped.')
else:
    summary = json.loads(summary_file.read_text(encoding='utf-8'))
    print('Source commit:', summary['source_commit'])
    print('Train class counts:', summary['train_class_counts'])
    print('Macro F1:', summary['macro_f1'])
    print('Files to download:')
    for file in sorted(output.iterdir()):
        print(' ', file)
